# ARC v0.20c — Full HNSW Mechanism Replication

**Goal.** Test whether approximation-feedback dynamics transfer, reverse, or remain stable under a graph-based ANN family while preserving the frozen FEVER/E5 feedback design.

This notebook is designed for a **fresh Google Colab runtime** and includes:

1. dependency bootstrap;
2. Google Drive mounting;
3. v0.18 corpus/query lineage audit;
4. v0.13 FIT/validation split audit against v0.18 sealed membership hashes;
5. FEVER DEV qrels loading;
6. HNSW index construction from the existing 55 E5 corpus shards;
7. FIT-only one-shot `efSearch` calibration;
8. deterministic low/high contrast freeze based only on one-shot FIT quality;
9. full 44-policy FIT and untouched-validation coupled trajectories;
10. H1/H2/H3abs/H3signed endpoints;
11. query-cluster bootstrap confidence intervals;
12. stable/amplifying/contracting regimes;
13. signed direction conditional on amplification;
14. FIT→validation configuration reproducibility;
15. threshold and alpha dose-response audits;
16. final report + SHA-256 provenance.

**No FEVER test qrels are accessed.**

### Important protocol note

This is a new post-v0.19 frozen extension. It does not retroactively alter any v0.19 claim gate. The HNSW contrast is selected only from FIT one-shot retrieval quality and is frozen before any feedback-trajectory outcome is computed.


In [ ]:
%pip install -q faiss-cpu==1.12.0 pyarrow psutil

from pathlib import Path
import os, json, hashlib, math, time, gc, platform, sys
import numpy as np
import pandas as pd
import faiss
import psutil

print("Python :", sys.version.split()[0])
print("FAISS  :", faiss.__version__)
print("NumPy  :", np.__version__)
print("Pandas :", pd.__version__)
print("RAM GiB:", round(psutil.virtual_memory().total / 2**30, 2))

assert faiss.__version__ == "1.12.0"
assert hasattr(faiss, "IndexHNSWFlat")
print("ENVIRONMENT CHECK: PASS")


## 1. Frozen configuration

The feedback grid matches the v0.18 E5 FEVER experiment. HNSW construction parameters are fixed here **before trajectory outcomes**.

`IndexHNSWFlat` is intentionally used so that low/high conditions share exactly the same graph and stored full-precision vectors; only `efSearch` changes.


In [ ]:
CONFIG = {
    "study_id": "ARC-v0.20c-HNSW-FULL",
    "seed": 20260820,
    "dim": 384,
    "top_k": 100,
    "utility_k": 10,
    "rounds": 4,
    "epsilon": 0.002,
    "epsilon_sensitivity": [0.0, 0.001, 0.002, 0.005, 0.01],

    # HNSW construction
    "M": 32,
    "efConstruction": 200,

    # FIT-only one-shot calibration ladder
    "ef_ladder": [8, 16, 32, 64, 128, 256],

    # Deterministic freeze rule.
    # HIGH = smallest ef within 0.5% relative of the best FIT nDCG@10.
    # LOW  = smallest lower ef with >=0.05 absolute nDCG@10 gap vs HIGH;
    #        if none exists, the smallest tested ef below HIGH.
    "high_relative_tolerance": 0.005,
    "low_min_abs_gap": 0.05,

    # Frozen feedback grid
    "alphas": [0.1, 0.3, 0.5, 0.7],
    "mean_k": [5, 20, 50],
    "softmax_k": [5, 20],
    "temperatures": [0.05, 0.1, 0.2, 0.5],

    # Inference
    "bootstrap_reps": 10000,
    "signed_bootstrap_reps": 10000,

    # Execution
    "checkpoint_every_queries": 25,
}

rng = np.random.default_rng(CONFIG["seed"])
print(json.dumps(CONFIG, indent=2))


## 2. Mount Drive and resolve existing ARC lineage


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/rag-pq-checkpoints")
ARC_ROOT = DRIVE_ROOT / "arc-v0"

# v0.18 E5 frozen corpus/query artifacts.
V018_CANDIDATES = [
    ARC_ROOT / "cross-encoder-fever-replication-v018" / "20260819-015645",
    ARC_ROOT / "cross-encoder-fever-replication-v018" / "20260819-100315",
    ARC_ROOT / "cross-encoder-fever-replication-v018" / "20260819-014003",
]
V018_RUN = next((p for p in V018_CANDIDATES
                 if (p/"corpus_encoding_manifest.json").exists()
                 and (p/"dev_query_embeddings.float32.npy").exists()), None)
if V018_RUN is None:
    raise FileNotFoundError("Could not resolve a complete v0.18 E5 run.")

V018_SHARDS = V018_RUN / "corpus_shards"
V018_QUERY_EMB = V018_RUN / "dev_query_embeddings.float32.npy"
V018_QUERY_IDS = V018_RUN / "dev_query_ids.txt"
V018_MANIFEST = V018_RUN / "corpus_encoding_manifest.json"
V018_PROTOCOL = V018_RUN / "v018_cross_encoder_protocol.json"

# v0.13 authoritative split. Prefer the continuation folder containing the persisted CSV.
V013_SPLIT_CANDIDATES = [
    ARC_ROOT / "fever-boundary-external-replication-v013" / "20260817-151852" / "v013_boundary_query_split.csv",
    ARC_ROOT / "fever-boundary-external-replication-v013" / "20260817-140640" / "v013_boundary_query_split.csv",
]
V013_SPLIT = next((p for p in V013_SPLIT_CANDIDATES if p.exists()), None)
if V013_SPLIT is None:
    raise FileNotFoundError("Could not resolve v013_boundary_query_split.csv")

FEVER_QRELS_DEV = DRIVE_ROOT / "raw-datasets" / "fever" / "qrels" / "dev.tsv"

HNSW_OUT = ARC_ROOT / "hnsw-mechanism-replication-v020c"
HNSW_OUT.mkdir(parents=True, exist_ok=True)

INDEX_PATH = HNSW_OUT / "fever-e5-small-v2-hnswflat-m32-efc200.faiss"
DOC_IDS_PATH = HNSW_OUT / "fever-e5-doc-ids.txt"
FREEZE_PATH = HNSW_OUT / "V020C_FROZEN_CONTRAST.json"

print("V018_RUN      :", V018_RUN)
print("V013_SPLIT    :", V013_SPLIT)
print("FEVER_QRELS   :", FEVER_QRELS_DEV)
print("OUTPUT        :", HNSW_OUT)

for p in [V018_SHARDS, V018_QUERY_EMB, V018_QUERY_IDS, V018_MANIFEST,
          V018_PROTOCOL, V013_SPLIT, FEVER_QRELS_DEV]:
    if not p.exists():
        raise FileNotFoundError(p)

print("DRIVE PATH CHECK: PASS")


## 3. Provenance helpers and full v0.18 lineage audit


In [ ]:
def sha256_file(path, chunk=8*1024*1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

def read_id_lines(path):
    with open(path, "r", encoding="utf-8") as f:
        return [line.rstrip("\n") for line in f]

def load_and_audit_lineage(full_hash_audit=True):
    protocol = json.loads(V018_PROTOCOL.read_text())
    manifest = json.loads(V018_MANIFEST.read_text())

    assert protocol["encoder"] == "intfloat/e5-small-v2"
    assert int(protocol["dimension"]) == 384
    assert int(protocol["feedback"]["rounds"]) == 4
    assert int(protocol["feedback"]["top_retrieve"]) == 100
    assert int(protocol["feedback"]["utility_k"]) == 10
    assert float(protocol["epsilon_primary"]) == 0.002
    assert protocol["test_accessed"] is False
    assert protocol["test_relevance_accessed"] is False

    assert manifest["status"] == "COMPLETE"
    assert int(manifest["rows"]) == 5_416_568
    assert len(manifest["shards"]) == 55
    assert sum(int(s["rows"]) for s in manifest["shards"]) == 5_416_568
    assert [int(s["shard"]) for s in manifest["shards"]] == list(range(55))

    bad = []
    for s in manifest["shards"]:
        sid = int(s["shard"])
        emb = V018_SHARDS / f"shard-{sid:04d}.float16.npy"
        ids = V018_SHARDS / f"shard-{sid:04d}.ids.txt"
        if not emb.exists() or not ids.exists():
            bad.append((sid, "missing file"))
            continue

        x = np.load(emb, mmap_mode="r")
        if x.shape != (int(s["rows"]), 384):
            bad.append((sid, f"shape={x.shape}"))
        if len(read_id_lines(ids)) != int(s["rows"]):
            bad.append((sid, "ID count mismatch"))

        if full_hash_audit:
            got = sha256_file(emb)
            if got != s["sha256"]:
                bad.append((sid, "embedding SHA256 mismatch"))

    if bad:
        raise RuntimeError(f"Lineage audit failed: {bad[:20]}")

    print("LINEAGE AUDIT: PASS")
    print("corpus rows:", manifest["rows"])
    print("shards     :", len(manifest["shards"]))
    print("encoder    :", protocol["encoder"])
    print("FIT hash   :", protocol["fit_membership_sha256"])
    print("VAL hash   :", protocol["validation_membership_sha256"])
    return protocol, manifest

protocol, corpus_manifest = load_and_audit_lineage(full_hash_audit=True)


## 4. Load queries, split, and DEV qrels


In [ ]:
def load_dev_queries():
    q = np.load(V018_QUERY_EMB, mmap_mode="r")
    qids = np.asarray(read_id_lines(V018_QUERY_IDS), dtype=object)
    assert q.shape == (6666, 384), q.shape
    assert len(qids) == 6666

    sample = np.asarray(q[np.linspace(0, len(q)-1, 512, dtype=int)], dtype=np.float32)
    norms = np.linalg.norm(sample, axis=1)
    if not np.allclose(norms, 1.0, atol=2e-3):
        raise ValueError("Query normalization audit failed.")
    return q, qids

def canonical_membership_hash(ids):
    payload = ("\n".join(sorted(map(str, ids))) + "\n").encode("utf-8")
    return hashlib.sha256(payload).hexdigest()

def load_split():
    s = pd.read_csv(V013_SPLIT)
    cols = {c.lower(): c for c in s.columns}

    q_candidates = ["query_id", "qid", "queryid"]
    split_candidates = ["split", "membership", "subset", "role"]

    qcol = next((cols[c] for c in q_candidates if c in cols), None)
    scol = next((cols[c] for c in split_candidates if c in cols), None)

    # Fallback: inspect string columns for values containing fit/val.
    if qcol is None:
        qcol = s.columns[0]
    if scol is None:
        for c in s.columns:
            vals = s[c].astype(str).str.lower()
            if vals.str.contains("fit").any() and vals.str.contains("val").any():
                scol = c
                break
    if scol is None:
        raise ValueError(f"Could not identify split column. Columns={list(s.columns)}")

    s[qcol] = s[qcol].astype(str)
    labels = s[scol].astype(str).str.lower()

    fit = set(s.loc[labels.str.contains("fit"), qcol])
    val = set(s.loc[labels.str.contains("val"), qcol])

    assert len(fit) == 3350, len(fit)
    assert len(val) == 3316, len(val)
    assert fit.isdisjoint(val)
    return fit, val, s

def audit_split(protocol, fit_ids, val_ids):
    got_fit = canonical_membership_hash(fit_ids)
    got_val = canonical_membership_hash(val_ids)
    exp_fit = protocol["fit_membership_sha256"]
    exp_val = protocol["validation_membership_sha256"]

    print("FIT:", got_fit)
    print("VAL:", got_val)
    assert got_fit == exp_fit, (got_fit, exp_fit)
    assert got_val == exp_val, (got_val, exp_val)
    print("SPLIT HASH AUDIT: PASS")

def load_qrels_dev():
    q = pd.read_csv(FEVER_QRELS_DEV, sep="\t")
    rename = {}
    for c in q.columns:
        z = c.lower().replace("_", "-")
        if z in {"query-id", "queryid", "qid"}:
            rename[c] = "query_id"
        elif z in {"corpus-id", "corpusid", "doc-id", "docid"}:
            rename[c] = "doc_id"
        elif z in {"score", "relevance", "rel"}:
            rename[c] = "relevance"
    q = q.rename(columns=rename)

    if not {"query_id", "doc_id", "relevance"} <= set(q.columns):
        q = pd.read_csv(FEVER_QRELS_DEV, sep="\t",
                        names=["query_id","doc_id","relevance"])

    q = q[["query_id","doc_id","relevance"]].copy()
    q["query_id"] = q["query_id"].astype(str)
    q["doc_id"] = q["doc_id"].astype(str)
    q["relevance"] = q["relevance"].astype(float)
    return q

queries, query_ids = load_dev_queries()
fit_ids, val_ids, split_df = load_split()
audit_split(protocol, fit_ids, val_ids)
qrels = load_qrels_dev()

print("queries:", queries.shape)
print("FIT    :", len(fit_ids))
print("VAL    :", len(val_ids))
print("qrels  :", len(qrels))
print("INPUT AUDIT: PASS")


## 5. RAM preflight for HNSWFlat

A 5.42M × 384 HNSWFlat index is large. The raw float32 vectors alone require about 7.75 GiB; graph and construction overhead add several more GiB.

This cell refuses to start a fresh build when total RAM is clearly insufficient. If an already-built index exists, it will be loaded instead.


In [ ]:
N_DOCS = 5_416_568
raw_vector_gib = N_DOCS * CONFIG["dim"] * 4 / 2**30
total_ram_gib = psutil.virtual_memory().total / 2**30
avail_ram_gib = psutil.virtual_memory().available / 2**30

print(f"Raw float32 vector storage ≈ {raw_vector_gib:.2f} GiB")
print(f"Total RAM              = {total_ram_gib:.2f} GiB")
print(f"Available RAM          = {avail_ram_gib:.2f} GiB")
print("Existing index         =", INDEX_PATH.exists())

if not INDEX_PATH.exists() and total_ram_gib < 18:
    raise MemoryError(
        "Fresh 5.42M-document IndexHNSWFlat build is unsafe in this runtime. "
        "Use a Colab High-RAM runtime (recommended >=18 GiB total RAM), "
        "then rerun from the top. This is a resource check, not an experimental failure."
    )

print("RAM PREFLIGHT: PASS")


## 6. Corpus shard iterator and HNSW index build/load


In [ ]:
def iter_corpus_shards(manifest):
    for s in manifest["shards"]:
        sid = int(s["shard"])
        embp = V018_SHARDS / f"shard-{sid:04d}.float16.npy"
        idsp = V018_SHARDS / f"shard-{sid:04d}.ids.txt"

        x16 = np.load(embp, mmap_mode="r")
        ids = read_id_lines(idsp)

        assert x16.shape == (int(s["rows"]), 384)
        assert len(ids) == len(x16)
        yield sid, x16, ids

def build_hnsw_from_v018_shards(manifest):
    index = faiss.IndexHNSWFlat(
        CONFIG["dim"],
        CONFIG["M"],
        faiss.METRIC_INNER_PRODUCT,
    )
    index.hnsw.efConstruction = CONFIG["efConstruction"]

    with open(DOC_IDS_PATH, "w", encoding="utf-8") as id_out:
        total = 0
        for sid, x16, ids in iter_corpus_shards(manifest):
            # Convert one shard at a time; do not concatenate the corpus in RAM.
            x = np.asarray(x16, dtype=np.float32)

            if not np.isfinite(x).all():
                raise ValueError(f"non-finite values in shard {sid}")

            sample = x[:min(2048, len(x))]
            norms = np.linalg.norm(sample, axis=1)
            if not np.allclose(norms, 1.0, atol=3e-3):
                raise ValueError(f"normalization audit failed in shard {sid}")

            index.add(x)
            id_out.write("\n".join(ids) + "\n")
            total += len(x)

            del x
            gc.collect()
            print(f"added shard {sid:02d}/54 | {total:,}/{N_DOCS:,}")

    assert index.ntotal == N_DOCS
    faiss.write_index(index, str(INDEX_PATH))

    record = {
        "study_id": CONFIG["study_id"],
        "index_type": "IndexHNSWFlat",
        "metric": "inner_product",
        "M": CONFIG["M"],
        "efConstruction": CONFIG["efConstruction"],
        "ntotal": int(index.ntotal),
        "source_v018_manifest_sha256": sha256_file(V018_MANIFEST),
        "source_v018_protocol_sha256": sha256_file(V018_PROTOCOL),
        "index_sha256": sha256_file(INDEX_PATH),
        "doc_ids_sha256": sha256_file(DOC_IDS_PATH),
        "faiss_version": faiss.__version__,
    }
    (HNSW_OUT/"v020c_hnsw_build_record.json").write_text(
        json.dumps(record, indent=2)
    )
    return index

if INDEX_PATH.exists():
    print("Loading existing HNSW index...")
    index = faiss.read_index(str(INDEX_PATH))
    assert index.ntotal == N_DOCS
else:
    print("Building HNSW index...")
    index = build_hnsw_from_v018_shards(corpus_manifest)

if not DOC_IDS_PATH.exists():
    # Reconstruct only the ID sidecar if needed.
    with open(DOC_IDS_PATH, "w", encoding="utf-8") as out_ids:
        n = 0
        for sid, _, ids in iter_corpus_shards(corpus_manifest):
            out_ids.write("\n".join(ids) + "\n")
            n += len(ids)
    assert n == N_DOCS

doc_ids = np.asarray(read_id_lines(DOC_IDS_PATH), dtype=object)
assert len(doc_ids) == N_DOCS
print("HNSW READY:", index.ntotal, "documents")


## 7. Retrieval metrics


In [ ]:
def hnsw_search(index, q, ef_search, k=None):
    k = CONFIG["top_k"] if k is None else int(k)
    index.hnsw.efSearch = int(ef_search)
    q = np.asarray(q, dtype=np.float32)
    return index.search(q, k)

def build_qrel_map(qrels_df):
    out = {}
    for qid, sub in qrels_df.groupby("query_id"):
        out[str(qid)] = {
            str(d): float(r)
            for d, r in zip(sub["doc_id"], sub["relevance"])
        }
    return out

QREL_MAP = build_qrel_map(qrels)

def dcg(rels):
    rels = np.asarray(rels, dtype=np.float64)
    if len(rels) == 0:
        return 0.0
    gains = np.power(2.0, rels) - 1.0
    discounts = 1.0 / np.log2(np.arange(2, len(rels) + 2))
    return float(np.sum(gains * discounts))

def ndcg_at_k(retrieved_doc_ids, qrel_dict, k=10):
    rels = [qrel_dict.get(str(d), 0.0) for d in retrieved_doc_ids[:k]]
    ideal = sorted(qrel_dict.values(), reverse=True)[:k]
    denom = dcg(ideal)
    return 0.0 if denom == 0 else dcg(rels) / denom

def query_mask(qids, selected):
    selected = set(map(str, selected))
    return np.asarray([str(x) in selected for x in qids], dtype=bool)


## 8. FIT-only one-shot `efSearch` ladder

This is the **only information used to select the low/high HNSW contrast**.
No feedback trajectory is computed before the next freeze cell.


In [ ]:
def run_fit_one_shot_ladder(index):
    mask = query_mask(query_ids, fit_ids)
    q = np.asarray(queries[mask], dtype=np.float32)
    qids = query_ids[mask]

    rows = []
    per_query_rows = []

    # Warm one tiny search to avoid including first-call overhead.
    _ = hnsw_search(index, q[:1], CONFIG["ef_ladder"][0], CONFIG["top_k"])

    for ef in CONFIG["ef_ladder"]:
        t0 = time.perf_counter()
        scores, I = hnsw_search(index, q, ef, CONFIG["top_k"])
        elapsed = time.perf_counter() - t0

        vals = []
        for i, qid in enumerate(qids):
            retrieved = [doc_ids[j] for j in I[i] if j >= 0]
            u = ndcg_at_k(retrieved, QREL_MAP.get(str(qid), {}), CONFIG["utility_k"])
            vals.append(u)
            per_query_rows.append({
                "query_id": str(qid),
                "efSearch": int(ef),
                "ndcg10": float(u),
            })

        row = {
            "efSearch": int(ef),
            "mean_ndcg10": float(np.mean(vals)),
            "latency_ms_per_query": elapsed * 1000.0 / len(q),
            "n_queries": int(len(q)),
        }
        rows.append(row)
        print(row)

    summary = pd.DataFrame(rows)
    per_query = pd.DataFrame(per_query_rows)

    summary.to_csv(HNSW_OUT/"v020c_fit_one_shot_ladder.csv", index=False)
    per_query.to_parquet(HNSW_OUT/"v020c_fit_one_shot_query_metrics.parquet", index=False)
    return summary, per_query

ladder, one_shot_query_metrics = run_fit_one_shot_ladder(index)
display(ladder)


## 9. Deterministic one-shot-only contrast freeze

The rule was fixed in the configuration cell:

- **HIGH:** smallest `efSearch` within 0.5% relative nDCG@10 of the best FIT ladder value.
- **LOW:** smallest tested lower `efSearch` whose absolute FIT nDCG@10 gap to HIGH is at least 0.05.
- If no lower point reaches that gap, use the smallest tested `efSearch` below HIGH.

This procedure does not inspect any feedback outcome.


In [ ]:
def choose_contrast_from_ladder(df):
    z = df.sort_values("efSearch").reset_index(drop=True).copy()
    best = float(z["mean_ndcg10"].max())
    threshold = best * (1.0 - CONFIG["high_relative_tolerance"])

    high_candidates = z[z["mean_ndcg10"] >= threshold]
    high_row = high_candidates.iloc[0]
    high_ef = int(high_row["efSearch"])
    high_ndcg = float(high_row["mean_ndcg10"])

    lower = z[z["efSearch"] < high_ef].copy()
    if len(lower) == 0:
        raise RuntimeError("No lower efSearch exists below selected HIGH.")

    eligible = lower[(high_ndcg - lower["mean_ndcg10"]) >= CONFIG["low_min_abs_gap"]]
    if len(eligible):
        low_row = eligible.sort_values("efSearch").iloc[0]
    else:
        low_row = lower.sort_values("efSearch").iloc[0]

    low_ef = int(low_row["efSearch"])
    low_ndcg = float(low_row["mean_ndcg10"])

    return {
        "low_efSearch": low_ef,
        "high_efSearch": high_ef,
        "low_fit_ndcg10": low_ndcg,
        "high_fit_ndcg10": high_ndcg,
        "absolute_gap": high_ndcg - low_ndcg,
        "selection_rule": {
            "high_relative_tolerance": CONFIG["high_relative_tolerance"],
            "low_min_abs_gap": CONFIG["low_min_abs_gap"],
        },
    }

freeze = choose_contrast_from_ladder(ladder)
LOW_EF = freeze["low_efSearch"]
HIGH_EF = freeze["high_efSearch"]

freeze.update({
    "study_id": CONFIG["study_id"],
    "selected_before_feedback_outcomes": True,
    "index_sha256": sha256_file(INDEX_PATH),
    "v018_protocol_sha256": sha256_file(V018_PROTOCOL),
    "v018_manifest_sha256": sha256_file(V018_MANIFEST),
    "fit_membership_sha256": protocol["fit_membership_sha256"],
    "validation_membership_sha256": protocol["validation_membership_sha256"],
    "faiss_version": faiss.__version__,
})

FREEZE_PATH.write_text(json.dumps(freeze, indent=2))
print(json.dumps(freeze, indent=2))
print("CONTRAST FROZEN:", LOW_EF, "vs", HIGH_EF)


## 10. Frozen 44-policy grid


In [ ]:
def make_policies():
    policies = []
    for alpha in CONFIG["alphas"]:
        for k in CONFIG["mean_k"]:
            policies.append({
                "family": "mean",
                "alpha": alpha,
                "k": k,
                "temperature": np.nan,
            })
        for k in CONFIG["softmax_k"]:
            for tau in CONFIG["temperatures"]:
                policies.append({
                    "family": "softmax",
                    "alpha": alpha,
                    "k": k,
                    "temperature": tau,
                })
    assert len(policies) == 44
    return policies

POLICIES = make_policies()
policy_df = pd.DataFrame(POLICIES)
display(policy_df)
print("policy count:", len(POLICIES))


## 11. Coupled trajectory implementation

The two branches share corpus, encoder, initial query, feedback policy, task, and evaluator. They differ only in HNSW `efSearch`.

Update:

\[
q_{t+1}^{r} = \mathrm{normalize}((1-\alpha)q_0 + \alpha F_t^{r}).
\]

Endpoints:

- `H1`: OLS slope of query-state cosine distance;
- `H2`: OLS slope of candidate-set Jaccard distance;
- `H3abs`: OLS slope of absolute nDCG@10 gap;
- `H3signed`: OLS slope of signed nDCG@10 gap (`high-low`).


In [ ]:
def normalize_vec(x, eps=1e-12):
    x = np.asarray(x, dtype=np.float32)
    n = float(np.linalg.norm(x))
    return x / max(n, eps)

def feedback_vector(doc_vecs, scores, family, temperature):
    x = np.asarray(doc_vecs, dtype=np.float32)

    if family == "mean":
        f = x.mean(axis=0)
    elif family == "softmax":
        tau = float(temperature)
        z = np.asarray(scores, dtype=np.float64) / tau
        z -= np.max(z)
        w = np.exp(z)
        w /= np.sum(w)
        f = (x * w[:, None]).sum(axis=0)
    else:
        raise ValueError(family)

    return normalize_vec(f)

def update_state(q0, f, alpha):
    return normalize_vec((1.0-alpha)*q0 + alpha*f)

def jaccard_distance(a, b):
    A = set(map(int, a))
    B = set(map(int, b))
    return 1.0 - len(A & B) / max(1, len(A | B))

def ols_slope(y):
    y = np.asarray(y, dtype=np.float64)
    x = np.arange(len(y), dtype=np.float64)
    xm = x.mean()
    ym = y.mean()
    return float(np.sum((x-xm)*(y-ym)) / np.sum((x-xm)**2))

# Feedback requires full corpus vectors for selected documents. We avoid loading
# all corpus vectors into RAM by building a row->(shard,offset) map arithmetically.
# Shards 0..53 contain 100,000 rows; shard 54 contains the remainder.
_SHARD_CACHE = {}

def get_corpus_rows(row_indices):
    row_indices = np.asarray(row_indices, dtype=np.int64)
    out = np.empty((len(row_indices), CONFIG["dim"]), dtype=np.float32)

    by_shard = {}
    for out_i, row in enumerate(row_indices):
        if row < 5_400_000:
            sid = int(row // 100_000)
            off = int(row % 100_000)
        else:
            sid = 54
            off = int(row - 5_400_000)
        by_shard.setdefault(sid, []).append((out_i, off))

    for sid, pairs in by_shard.items():
        if sid not in _SHARD_CACHE:
            p = V018_SHARDS / f"shard-{sid:04d}.float16.npy"
            _SHARD_CACHE[sid] = np.load(p, mmap_mode="r")

        mm = _SHARD_CACHE[sid]
        outs = [a for a, _ in pairs]
        offs = [b for _, b in pairs]
        out[outs] = np.asarray(mm[offs], dtype=np.float32)

    return out

def run_single_pair(q0, qid, policy):
    q0 = normalize_vec(q0)
    qL = q0.copy()
    qH = q0.copy()

    d_hist, j_hist, a_hist, g_hist = [], [], [], []
    uL_hist, uH_hist = [], []

    for t in range(CONFIG["rounds"] + 1):
        sL, iL = hnsw_search(index, qL[None,:], LOW_EF, CONFIG["top_k"])
        sH, iH = hnsw_search(index, qH[None,:], HIGH_EF, CONFIG["top_k"])
        sL, iL = sL[0], iL[0]
        sH, iH = sH[0], iH[0]

        validL = iL >= 0
        validH = iH >= 0
        iLv, sLv = iL[validL], sL[validL]
        iHv, sHv = iH[validH], sH[validH]

        docsL = [doc_ids[j] for j in iLv]
        docsH = [doc_ids[j] for j in iHv]

        qrel = QREL_MAP.get(str(qid), {})
        uL = ndcg_at_k(docsL, qrel, CONFIG["utility_k"])
        uH = ndcg_at_k(docsH, qrel, CONFIG["utility_k"])

        d_hist.append(1.0 - float(np.dot(qL, qH)))
        j_hist.append(jaccard_distance(iLv, iHv))
        a_hist.append(abs(uH-uL))
        g_hist.append(uH-uL)
        uL_hist.append(uL)
        uH_hist.append(uH)

        if t == CONFIG["rounds"]:
            break

        k = int(policy["k"])
        if len(iLv) < k or len(iHv) < k:
            raise RuntimeError("Fewer than k valid retrieval results.")

        vecL = get_corpus_rows(iLv[:k])
        vecH = get_corpus_rows(iHv[:k])

        temp = policy["temperature"]
        fL = feedback_vector(vecL, sLv[:k], policy["family"], temp)
        fH = feedback_vector(vecH, sHv[:k], policy["family"], temp)

        qL = update_state(q0, fL, policy["alpha"])
        qH = update_state(q0, fH, policy["alpha"])

    return {
        "query_id": str(qid),
        "family": policy["family"],
        "alpha": float(policy["alpha"]),
        "k": int(policy["k"]),
        "temperature": float(policy["temperature"]) if not pd.isna(policy["temperature"]) else np.nan,
        "H1": ols_slope(d_hist),
        "H2": ols_slope(j_hist),
        "H3abs": ols_slope(a_hist),
        "H3signed": ols_slope(g_hist),
        "final_uL": float(uL_hist[-1]),
        "final_uH": float(uH_hist[-1]),
        "final_signed_gap": float(uH_hist[-1]-uL_hist[-1]),
        "d0": d_hist[0], "d1": d_hist[1], "d2": d_hist[2], "d3": d_hist[3], "d4": d_hist[4],
        "j0": j_hist[0], "j1": j_hist[1], "j2": j_hist[2], "j3": j_hist[3], "j4": j_hist[4],
        "a0": a_hist[0], "a1": a_hist[1], "a2": a_hist[2], "a3": a_hist[3], "a4": a_hist[4],
        "g0": g_hist[0], "g1": g_hist[1], "g2": g_hist[2], "g3": g_hist[3], "g4": g_hist[4],
    }


## 12. Resumable full split runner

This writes one checkpoint parquet per block of queries. If Colab disconnects, rerunning the cell resumes from completed blocks.


In [ ]:
def run_split_resumable(split_name, selected_ids):
    split_dir = HNSW_OUT / split_name
    split_dir.mkdir(parents=True, exist_ok=True)

    mask = query_mask(query_ids, selected_ids)
    q = np.asarray(queries[mask], dtype=np.float32)
    qids = query_ids[mask]

    expected_n = len(q)
    print(split_name, "queries:", expected_n)

    chunk = CONFIG["checkpoint_every_queries"]

    for start in range(0, expected_n, chunk):
        stop = min(start + chunk, expected_n)
        cp = split_dir / f"{split_name}_{start:04d}_{stop:04d}.parquet"

        if cp.exists():
            print("skip existing", cp.name)
            continue

        rows = []
        t0 = time.perf_counter()

        for qi in range(start, stop):
            q0 = q[qi]
            qid = qids[qi]
            for policy in POLICIES:
                rows.append(run_single_pair(q0, qid, policy))

        tmp = cp.with_suffix(".tmp.parquet")
        pd.DataFrame(rows).to_parquet(tmp, index=False)
        os.replace(tmp, cp)

        dt = time.perf_counter() - t0
        print(f"wrote {cp.name}: {len(rows):,} events | {dt:.1f}s")

    parts = sorted(split_dir.glob(f"{split_name}_*.parquet"))
    df = pd.concat([pd.read_parquet(p) for p in parts], ignore_index=True)

    expected_events = expected_n * 44
    assert len(df) == expected_events, (len(df), expected_events)
    assert df["query_id"].nunique() == expected_n

    merged = HNSW_OUT / f"v020c_{split_name}_endpoints.parquet"
    df.to_parquet(merged, index=False)
    print("merged:", merged, "rows:", len(df))
    return df


## 13. Run FIT trajectories

This happens **after** the HNSW contrast is frozen.


In [ ]:
fit_df = run_split_resumable("fit", fit_ids)
print(fit_df.shape)
display(fit_df.head())


## 14. Run untouched validation trajectories


In [ ]:
val_df = run_split_resumable("validation", val_ids)
print(val_df.shape)
display(val_df.head())


## 15. Query-cluster bootstrap aggregate endpoints

The query is the independent sampling unit. All 44 policy realizations of each query remain together.


In [ ]:
def query_level_means(df, metric):
    return df.groupby("query_id", sort=False)[metric].mean().to_numpy(dtype=np.float64)

def bootstrap_mean_ci(df, metric, reps=None, seed=None):
    reps = CONFIG["bootstrap_reps"] if reps is None else int(reps)
    seed = CONFIG["seed"] if seed is None else int(seed)

    vals = query_level_means(df, metric)
    obs = float(vals.mean())

    r = np.random.default_rng(seed)
    boot = np.empty(reps, dtype=np.float64)
    n = len(vals)

    for b in range(reps):
        idx = r.integers(0, n, size=n)
        boot[b] = vals[idx].mean()

    lo, hi = np.quantile(boot, [0.025, 0.975])
    return {
        "metric": metric,
        "mean": obs,
        "ci_low": float(lo),
        "ci_high": float(hi),
        "bootstrap_reps": reps,
        "n_queries": n,
    }

def endpoint_summary(df):
    return pd.DataFrame([
        bootstrap_mean_ci(df, m, seed=CONFIG["seed"] + i)
        for i, m in enumerate(["H1","H2","H3abs","H3signed"])
    ])

fit_endpoint_summary = endpoint_summary(fit_df)
val_endpoint_summary = endpoint_summary(val_df)

fit_endpoint_summary["split"] = "FIT"
val_endpoint_summary["split"] = "validation"

endpoint_all = pd.concat([fit_endpoint_summary, val_endpoint_summary], ignore_index=True)
endpoint_all.to_csv(HNSW_OUT/"v020c_aggregate_endpoints.csv", index=False)
display(endpoint_all)


## 16. Regime prevalence and epsilon sensitivity


In [ ]:
def classify_regime(x, eps):
    if x > eps:
        return "amplifying"
    if x < -eps:
        return "contracting"
    return "stable_null"

def regime_summary(df, eps):
    reg = df["H3abs"].map(lambda x: classify_regime(x, eps))
    counts = reg.value_counts()
    n = len(reg)
    return {
        "epsilon": eps,
        "stable_null_pct": 100.0 * counts.get("stable_null", 0) / n,
        "amplifying_pct": 100.0 * counts.get("amplifying", 0) / n,
        "contracting_pct": 100.0 * counts.get("contracting", 0) / n,
        "n_events": n,
    }

threshold_rows = []
for split_name, df in [("FIT", fit_df), ("validation", val_df)]:
    for eps in CONFIG["epsilon_sensitivity"]:
        r = regime_summary(df, eps)
        r["split"] = split_name
        threshold_rows.append(r)

threshold_df = pd.DataFrame(threshold_rows)
threshold_df.to_csv(HNSW_OUT/"v020c_regime_threshold_sensitivity.csv", index=False)
display(threshold_df)


## 17. Signed direction conditional on amplification


In [ ]:
def signed_direction_summary(df, eps=0.002, reps=None, seed=None):
    reps = CONFIG["signed_bootstrap_reps"] if reps is None else reps
    seed = CONFIG["seed"] + 100 if seed is None else seed

    x = df[df["H3abs"] > eps].copy()
    if len(x) == 0:
        raise RuntimeError("No amplification events.")

    x["higher_win"] = (x["final_signed_gap"] > 0).astype(float)
    x["lower_win"] = (x["final_signed_gap"] < 0).astype(float)
    x["tie"] = (x["final_signed_gap"] == 0).astype(float)

    qids = x["query_id"].drop_duplicates().to_numpy()
    grouped = {
        qid: x.loc[x["query_id"] == qid, "higher_win"].to_numpy(dtype=float)
        for qid in qids
    }

    obs = float(x["higher_win"].mean())
    r = np.random.default_rng(seed)
    boot = np.empty(reps, dtype=np.float64)

    n = len(qids)
    for b in range(reps):
        sampled = qids[r.integers(0, n, size=n)]
        # preserve each sampled query's full set of qualifying events
        vals = [grouped[q] for q in sampled]
        boot[b] = np.concatenate(vals).mean()

    lo, hi = np.quantile(boot, [0.025, 0.975])

    return {
        "epsilon": eps,
        "n_amplification_events": int(len(x)),
        "n_affected_queries": int(x["query_id"].nunique()),
        "higher_win_fraction": obs,
        "higher_win_ci_low": float(lo),
        "higher_win_ci_high": float(hi),
        "lower_win_fraction": float(x["lower_win"].mean()),
        "tie_fraction": float(x["tie"].mean()),
        "bootstrap_reps": reps,
    }

signed_val = signed_direction_summary(val_df, CONFIG["epsilon"])
pd.DataFrame([signed_val]).to_csv(
    HNSW_OUT/"v020c_validation_signed_direction.csv", index=False
)
display(pd.DataFrame([signed_val]))


## 18. Alpha dose response


In [ ]:
alpha_rows = []
for alpha, sub in val_df.groupby("alpha"):
    reg = regime_summary(sub, CONFIG["epsilon"])
    alpha_rows.append({
        "alpha": float(alpha),
        "amplification_pct": reg["amplifying_pct"],
        "stable_null_pct": reg["stable_null_pct"],
        "contracting_pct": reg["contracting_pct"],
        "mean_H3abs": float(sub["H3abs"].mean()),
        "mean_H3signed": float(sub["H3signed"].mean()),
    })

alpha_df = pd.DataFrame(alpha_rows).sort_values("alpha")
alpha_df.to_csv(HNSW_OUT/"v020c_validation_alpha_dose_response.csv", index=False)
display(alpha_df)


## 19. FIT → validation configuration reproducibility


In [ ]:
CONFIG_COLS = ["family","alpha","k","temperature"]

def amplification_by_config(df, eps):
    z = df.copy()
    z["temperature_key"] = z["temperature"].fillna(-1.0)
    z["amplify"] = z["H3abs"] > eps
    g = (z.groupby(["family","alpha","k","temperature_key"], as_index=False)
           ["amplify"].mean())
    return g

fit_cfg = amplification_by_config(fit_df, CONFIG["epsilon"]).rename(columns={"amplify":"fit"})
val_cfg = amplification_by_config(val_df, CONFIG["epsilon"]).rename(columns={"amplify":"val"})
cfg = fit_cfg.merge(val_cfg, on=["family","alpha","k","temperature_key"], how="inner")

pearson = cfg["fit"].corr(cfg["val"], method="pearson")
spearman = cfg["fit"].corr(cfg["val"], method="spearman")

cfg["fit_centered"] = cfg["fit"] - cfg.groupby("alpha")["fit"].transform("mean")
cfg["val_centered"] = cfg["val"] - cfg.groupby("alpha")["val"].transform("mean")

pearson_centered = cfg["fit_centered"].corr(cfg["val_centered"], method="pearson")
spearman_centered = cfg["fit_centered"].corr(cfg["val_centered"], method="spearman")

corr_report = {
    "pearson": float(pearson),
    "spearman": float(spearman),
    "alpha_centered_pearson": float(pearson_centered),
    "alpha_centered_spearman": float(spearman_centered),
}

cfg.to_csv(HNSW_OUT/"v020c_configuration_reproducibility.csv", index=False)
(HNSW_OUT/"v020c_configuration_correlations.json").write_text(
    json.dumps(corr_report, indent=2)
)

print(json.dumps(corr_report, indent=2))
display(cfg)


## 20. Frozen interpretation gate

The result is classified without retuning:

- **TRANSFER:** validation 95% CI for mean H3abs is entirely above zero.
- **REVERSAL:** validation 95% CI for mean H3abs is entirely below zero.
- **NULL/STABLE:** validation H3abs CI contains zero.

This is deliberately outcome-neutral.


In [ ]:
h3 = val_endpoint_summary[val_endpoint_summary["metric"] == "H3abs"].iloc[0]

if h3["ci_low"] > 0:
    mechanism_gate = "HNSW_TRANSFER"
elif h3["ci_high"] < 0:
    mechanism_gate = "HNSW_REVERSAL"
else:
    mechanism_gate = "HNSW_NULL_OR_STABLE"

print("MECHANISM GATE:", mechanism_gate)


## 21. Final audit report and hashes


In [ ]:
def maybe_hash(path):
    return sha256_file(path) if Path(path).exists() else None

final_report = {
    "study_id": CONFIG["study_id"],
    "mechanism_gate": mechanism_gate,
    "low_efSearch": LOW_EF,
    "high_efSearch": HIGH_EF,
    "fit_one_shot": ladder.to_dict(orient="records"),
    "fit_endpoints": fit_endpoint_summary.to_dict(orient="records"),
    "validation_endpoints": val_endpoint_summary.to_dict(orient="records"),
    "validation_primary_regime": regime_summary(val_df, CONFIG["epsilon"]),
    "validation_signed_direction": signed_val,
    "configuration_reproducibility": corr_report,
    "alpha_dose_response": alpha_df.to_dict(orient="records"),
    "frozen_lineage": {
        "encoder": protocol["encoder"],
        "dimension": protocol["dimension"],
        "fit_membership_sha256": protocol["fit_membership_sha256"],
        "validation_membership_sha256": protocol["validation_membership_sha256"],
        "v018_protocol_sha256": sha256_file(V018_PROTOCOL),
        "v018_manifest_sha256": sha256_file(V018_MANIFEST),
        "hnsw_index_sha256": sha256_file(INDEX_PATH),
        "doc_ids_sha256": sha256_file(DOC_IDS_PATH),
        "contrast_freeze_sha256": sha256_file(FREEZE_PATH),
    },
    "test_accessed": False,
    "test_relevance_accessed": False,
    "software": {
        "faiss": faiss.__version__,
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "python": sys.version.split()[0],
    },
}

REPORT_PATH = HNSW_OUT / "v020c_final_report.json"
REPORT_PATH.write_text(json.dumps(final_report, indent=2))

artifact_paths = [
    FREEZE_PATH,
    HNSW_OUT/"v020c_fit_one_shot_ladder.csv",
    HNSW_OUT/"v020c_fit_one_shot_query_metrics.parquet",
    HNSW_OUT/"v020c_fit_endpoints.parquet",
    HNSW_OUT/"v020c_validation_endpoints.parquet",
    HNSW_OUT/"v020c_aggregate_endpoints.csv",
    HNSW_OUT/"v020c_regime_threshold_sensitivity.csv",
    HNSW_OUT/"v020c_validation_signed_direction.csv",
    HNSW_OUT/"v020c_validation_alpha_dose_response.csv",
    HNSW_OUT/"v020c_configuration_reproducibility.csv",
    HNSW_OUT/"v020c_configuration_correlations.json",
    REPORT_PATH,
]

hash_rows = []
for p in artifact_paths:
    if p.exists():
        hash_rows.append({
            "file": p.name,
            "sha256": sha256_file(p),
            "bytes": p.stat().st_size,
        })

hash_df = pd.DataFrame(hash_rows)
hash_df.to_csv(HNSW_OUT/"V020C_ARTIFACT_SHA256.csv", index=False)

print(json.dumps(final_report, indent=2)[:6000])
print("\nFINAL REPORT:", REPORT_PATH)
display(hash_df)


# Completion criteria

A scientifically complete v0.20c run should leave the following in:

`/content/drive/MyDrive/rag-pq-checkpoints/arc-v0/hnsw-mechanism-replication-v020c/`

including:

- HNSW index and doc-ID sidecar;
- FIT one-shot ladder;
- immutable frozen contrast JSON;
- full FIT endpoints;
- full untouched-validation endpoints;
- aggregate endpoint bootstrap CIs;
- regime threshold sensitivity;
- signed-direction audit;
- alpha dose response;
- configuration reproducibility;
- final report;
- SHA-256 artifact manifest.

Do **not** change `LOW_EF`, `HIGH_EF`, the feedback grid, epsilon, or the statistical unit after feedback outcomes have been computed.
